In [ ]:
import xgi as xgi
import matplotlib as plt
import pandas as pd

In [ ]:
# Step 1: Create a hypergraph
H = xgi.load_xgi_data("senate-committees")

In [ ]:
# Step 2: Print measurable properties
print("=== Basic Properties ===")
print("Nodes:", list(H.nodes))
print("Edges:", list(H.edges))
print("Number of nodes:", H.num_nodes)
print("Number of edges:", H.num_edges)

print("\n=== Node Degrees ===")
for node in H.nodes:
    # Check if the node has a 'label' attribute before accessing it
    label = H.nodes[node].get('label', 'N/A') # Use 'N/A' if 'label' is not found
    print(f"Node {node} ({label}): Degree = {H.degree(node)}")

print("\n=== Edge Sizes ===")
for edge in H.edges:
    print(f"Edge {edge}: Size = {H.size(edge)}")

In [ ]:
# Step 3: Visualization
# Use the spiral layout
pos = xgi.layout.spiral_layout(H)

# Draw the hypergraph with node labels
plt.figure(figsize=(8, 6))
xgi.draw(H, pos=pos, with_node_labels=True, with_edge_labels=True)

# Adding custom node labels
node_labels = {n: H.nodes[n].get("label", str(n)) for n in H.nodes}
for n, (x, y) in pos.items():
    plt.text(x, y + 0.05, node_labels[n], fontsize=10, ha="center", color="blue")

plt.title("Hypergraph Visualization")

# Step 4: Additional Measurements
print("\n=== Additional Measurements ===")
print("Weighted Degree of Nodes:")
for node in H.nodes:
    # Use H.nodes.memberships(node) to get edge IDs
    weighted_degree = sum(H.edges[edge].get("weight", 1) for edge in H.nodes.memberships(node))
    print(f"Node {node} ({H.nodes[node].get('label', str(node))}): Weighted Degree = {weighted_degree}")
print("\nEdge Weights:")
for edge in H.edges:
    weight = H.edges[edge].get("weight", "N/A")
    print(f"Edge {edge}: Weight = {weight}")

# Step 5: Hyperdegree Centrality Calculation
print("\n=== Hyperdegree Centrality ===")
for node in H.nodes:
    hyperdegree = H.degree(node)
    label = H.nodes[node].get('label', str(node))
    print(f"Node {node} ({label}): Hyperdegree Centrality = {hyperdegree}")

# follow up on assortativity, centraliities, connectedness, clustering coefficient, desnity

print(xgi.clustering_coefficient(H))

# show graph
plt.show()

In [ ]:
# try exporting into bipartite graph and displaying in igraph
edge_list = xgi.to_bipartite_edgelist(H);

filepath = "/content/drive/My Drive/bipartite_edges.csv"

# save bipartite edge_list as csv
with open(filepath, "w") as f:

    f.write("node, hyperedge\n")

    for edge in edge_list:
        f.write(f"{edge[0]},{edge[1]}\n")

In [ ]:
# calculate centralities
# H_eigienvector_centrality = xgi.h_eigenvector_centrality(H)

Z_eigenvector_centrality = xgi.z_eigenvector_centrality(H)

# C_eigenvector_centrality = xgi.clique_eigenvector_centrality(H)

In [ ]:
centralities_dict = {}
for i in range(1, len(H_eigienvector_centrality)):
  centralities = []
  centralities.append(H_eigienvector_centrality.get(str(i)))
  centralities.append(Z_eigenvector_centrality.get(str(i)))
  centralities.append(C_eigenvector_centrality.get(str(i)))
  centralities_dict[i] = centralities

# dictionary of the three eigenvector centrality calculations for each node
# [H_eigen, Z_eigen, Clique]
print (centralities_dict)

### H-Eigenvector Centrality

In [ ]:
# Load hypergraph from edge list (same as in R)
edges_df = pd.read_csv("bipartite_edges.csv")

# Convert to incidence dict: hyperedge -> list of nodes
from collections import defaultdict
Hdict = defaultdict(list)
for _, row in edges_df.iterrows():
    Hdict[row['hyperedge']].append(row['node'])

# Build hypergraph
H = xgi.Hypergraph(Hdict)

# Compute HEC centrality in XGI
hec_py = xgi.algorithms.centrality.h_eigenvector_centrality(H)

# Convert to DataFrame
hec_py_df = pd.DataFrame(list(hec_py.items()), columns=["node", "cec_py"])

hec_r_df = pd.read_csv("hec_scores_r (4).csv", dtype={'node': str})

print(hec_r_df)

In [ ]:
# compare R file to Python file

# Rename the columns to have desired names
hec_r_df = hec_r_df.drop(columns=hec_r_df.columns[0])  # Drop the first column (index)
hec_r_df.columns = ['node', 'hec_r'] # Assuming the second column is 'hec_r'

# convert node columns in both to string
hec_r_df["node"] = hec_r_df["node"].astype(str)
hec_py_df["node"] = hec_py_df["node"].astype(str)
merged_df = pd.merge(hec_r_df, hec_py_df, on="node", how="inner")

# Merge on node ID (make sure formats match)
merged_df = pd.merge(hec_r_df, hec_py_df, on="node", how="inner")

# Compare: correlation, scatterplot
print(merged_df.corr())

import matplotlib.pyplot as plt

plt.scatter(merged_df["hec_r"], merged_df["hec_py"], alpha=0.6)
plt.xlabel("HEC from R")
plt.ylabel("HEC from Python (XGI)")
plt.title("HEC Score Comparison: R vs Python")
plt.grid(True)
plt.show()

### Z-Eigenvector Centrality

In [ ]:
# Load hypergraph from edge list (same as in R)
edges_df = pd.read_csv("bipartite_edges.csv")

# Convert to incidence dict: hyperedge -> list of nodes
from collections import defaultdict
Hdict = defaultdict(list)
for _, row in edges_df.iterrows():
    Hdict[row['hyperedge']].append(row['node'])

# Build hypergraph
H = xgi.Hypergraph(Hdict)

# Compute ZEC centrality in XGI
zec_py = xgi.algorithms.centrality.z_eigenvector_centrality(H)

# Convert to DataFrame
zec_py_df = pd.DataFrame(list(zec_py.items()), columns=["node", "cec_py"])